# Ensemble (Chapter 6)

This notebook accompanies Chapter 6 §6.9 of *Context Engineering with DSPy*. It builds three small optimized programs (LabeledFewShot, BootstrapFewShot, BootstrapFewShotWithRandomSearch), then combines them via `dspy.teleprompt.Ensemble` with majority voting and a size-limited variant.

Total cost is around $1–2 (the BootstrapRS leg is the dominant one).


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import dspy
import pandas as pd
import time
import os
import random
from dotenv import load_dotenv

load_dotenv()

# Pricing per 1M tokens (as of Dec 2025)
PRICING = {
    "openai/gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "openai/gpt-4o": {"input": 2.50, "output": 10.00},
}

NUM_THREADS = 16

def calculate_cost(usage: dict) -> float:
    total_cost = 0.0
    for model, data in usage.items():
        if model in PRICING:
            prompt_tokens = data.get("prompt_tokens", 0)
            completion_tokens = data.get("completion_tokens", 0)
            total_cost += (prompt_tokens / 1_000_000) * PRICING[model]["input"]
            total_cost += (completion_tokens / 1_000_000) * PRICING[model]["output"]
    return total_cost

In [ ]:
# Initialize models
task_lm = dspy.LM(
    "openai/gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=16000,
    cache=False,
)

dspy.configure(lm=task_lm)

In [ ]:
# Load dataset
csv_path = '../data/ai_vs_human200.csv'
df = pd.read_csv(csv_path)
examples = df.to_dict(orient='records')

dataset = [
    dspy.Example(**ex).with_inputs("text")
    for ex in examples
]

# Same split as optimizer-benchmark-v2.ipynb
random.seed(42)
random.shuffle(dataset)

n = len(dataset)
train_end = int(n * 0.5)
val_end = int(n * 0.75)

trainset = dataset[:train_end]
valset = dataset[train_end:val_end]
testset = dataset[val_end:]

print(f"Train: {len(trainset)}, Val: {len(valset)}, Test: {len(testset)}")

In [ ]:
# Define module and metric (same as other notebooks)
class DetectAIText(dspy.Signature):
    """Detects whether text is AI-generated."""
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="Whether the text is AI-generated")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def exact_match(example, response, trace=None):
    return 1 if example.is_ai == response.is_ai else 0

## Step 1: Build Individual Optimized Programs

Ensembles work best when the component programs are **diverse** — optimized with different
strategies so they make different mistakes. We'll use three optimizers that take very
different approaches:

1. **LabeledFewShot** — random examples from the training set (no bootstrapping)
2. **BootstrapFewShot** — bootstrapped traces from successful runs
3. **BootstrapFewShotWithRandomSearch** — searches over multiple demo configurations

In [ ]:
from dspy.teleprompt import LabeledFewShot, BootstrapFewShot, BootstrapFewShotWithRandomSearch

evaluator = dspy.Evaluate(
    devset=testset,
    metric=exact_match,
    num_threads=NUM_THREADS,
    display_progress=True,
    display_table=False,
)

In [ ]:
# Program 1: LabeledFewShot
print("=" * 60)
print("Optimizing Program 1: LabeledFewShot")
print("=" * 60)

optimizer1 = LabeledFewShot(k=4)
program_labeled = optimizer1.compile(AIDetector(), trainset=trainset)

result1 = evaluator(program_labeled)
print(f"LabeledFewShot accuracy: {result1.score:.1f}%")

In [ ]:
# Program 2: BootstrapFewShot
print("=" * 60)
print("Optimizing Program 2: BootstrapFewShot")
print("=" * 60)

optimizer2 = BootstrapFewShot(
    metric=exact_match,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
    max_rounds=1,
)
program_bootstrap = optimizer2.compile(AIDetector(), trainset=trainset)

result2 = evaluator(program_bootstrap)
print(f"BootstrapFewShot accuracy: {result2.score:.1f}%")

In [ ]:
# Program 3: BootstrapFewShotWithRandomSearch
print("=" * 60)
print("Optimizing Program 3: BootstrapFewShotWithRandomSearch")
print("=" * 60)

optimizer3 = BootstrapFewShotWithRandomSearch(
    metric=exact_match,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
    max_rounds=1,
    num_candidate_programs=8,
    num_threads=NUM_THREADS,
)
program_random_search = optimizer3.compile(
    AIDetector(), trainset=trainset, valset=valset
)

result3 = evaluator(program_random_search)
print(f"BootstrapFewShotWithRandomSearch accuracy: {result3.score:.1f}%")

In [ ]:
print("\nIndividual Program Results:")
print(f"  LabeledFewShot:                    {result1.score:.1f}%")
print(f"  BootstrapFewShot:                  {result2.score:.1f}%")
print(f"  BootstrapFewShotWithRandomSearch:   {result3.score:.1f}%")

## Step 2: Ensemble with Majority Voting

Now we combine all three programs using `dspy.Ensemble` with `dspy.majority`.
For each input, the ensemble runs all three programs and returns whichever
answer appears most often.

In [ ]:
from dspy.teleprompt import Ensemble

# Ensemble with majority voting
ensemble_optimizer = Ensemble(reduce_fn=dspy.majority)
ensemble_program = ensemble_optimizer.compile(
    [program_labeled, program_bootstrap, program_random_search]
)

print("Ensemble compiled with 3 programs using dspy.majority voting")

In [ ]:
# Evaluate the ensemble on the test set and track cost
print("Evaluating ensemble on test set...")

start_time = time.time()
with dspy.track_usage() as usage:
    ensemble_result = evaluator(ensemble_program)
elapsed = time.time() - start_time

total_usage = usage.get_total_tokens()
total_tokens = sum(d.get("total_tokens", 0) for d in total_usage.values())
cost = calculate_cost(total_usage)

print(f"\n{'=' * 60}")
print(f"ENSEMBLE RESULTS")
print(f"{'=' * 60}")
print(f"Ensemble accuracy:   {ensemble_result.score:.1f}%")
print(f"Best individual:     {max(result1.score, result2.score, result3.score):.1f}%")
print(f"Tokens used:         {total_tokens:,}")
print(f"Cost (eval only):    ${cost:.4f}")
print(f"Time:                {elapsed:.1f}s")
print(f"{'=' * 60}")

## Step 3: Ensemble with Size Limitation

You can also limit how many programs run per query using the `size` parameter.
This randomly samples a subset of programs each time, reducing cost at the
expense of some variance.

In [ ]:
# Ensemble with size=2 (randomly samples 2 of 3 programs per query)
ensemble_sampled = Ensemble(reduce_fn=dspy.majority, size=2)
ensemble_sampled_program = ensemble_sampled.compile(
    [program_labeled, program_bootstrap, program_random_search]
)

print("Evaluating sampled ensemble (size=2) on test set...")

start_time = time.time()
with dspy.track_usage() as usage:
    sampled_result = evaluator(ensemble_sampled_program)
elapsed = time.time() - start_time

total_usage = usage.get_total_tokens()
sampled_tokens = sum(d.get("total_tokens", 0) for d in total_usage.values())
sampled_cost = calculate_cost(total_usage)

print(f"\nSampled ensemble (size=2) accuracy: {sampled_result.score:.1f}%")
print(f"Tokens: {sampled_tokens:,}  Cost: ${sampled_cost:.4f}  Time: {elapsed:.1f}s")

## Final Comparison

In [ ]:
# Summary table
summary = pd.DataFrame([
    {
        "Program": "LabeledFewShot",
        "Accuracy": f"{result1.score:.1f}%",
        "LM Calls": "1× per query",
    },
    {
        "Program": "BootstrapFewShot",
        "Accuracy": f"{result2.score:.1f}%",
        "LM Calls": "1× per query",
    },
    {
        "Program": "BootstrapFewShotWithRandomSearch",
        "Accuracy": f"{result3.score:.1f}%",
        "LM Calls": "1× per query",
    },
    {
        "Program": "Ensemble (all 3, majority vote)",
        "Accuracy": f"{ensemble_result.score:.1f}%",
        "LM Calls": "3× per query",
    },
    {
        "Program": "Ensemble (size=2, majority vote)",
        "Accuracy": f"{sampled_result.score:.1f}%",
        "LM Calls": "2× per query",
    },
])

print(summary.to_string(index=False))